# AIF Records with yabadaba

This Notebook provides an overview and examples of how AIF records are integrated with yabadaba and the basic features this provides.  See the next Notebook for more complex database features.

## 1. What does yabadaba do?

The yabadaba package defines generic classes for **Database** interactions and data **Record** interpretations and transformations.  The yabadaba package itself also contains Database subclasses that then implement the different database interactions with specific database architectures.  Record subclasses are not included in the core yabadaba package as they should be defined for specific types of data.  The Record subclasses for AIF files are therefore part of the seperate yabadaba_aif package contained in this repository.

The default Database class has methods to get, count, add, delete and update data records and to store supporting raw data files in the database associated with the data entries.  There are also three built-in subclasses supporting MongoDB databases, MDCS curator databases, and a minimum infrastructure local database consisting only of JSON/XML and csv files.

The default Record class manages data entry values and supports transformations between the Python object representation, tree-like data model representation for JSON/XML, and a flat dict representation containing only simple value fields.

### 1.1. What does yabadaba_aif do?

1. Defines subclasses of yabadaba Record called AIF and AIF2 that specify the values in the AIF schema and add transformation methods to/from the AIF format.
2. Defines a subclass of yabadaba Database called AIFDatabase that behaves the same as the local database style but allows for the files to be stored in the AIF format.
3. The init file then integrates the new subclasses into yabadaba.

### 1.2. What are in the AIF and AIF2 record definitions?

AIF and AIF2 are nearly identical except that AIF represents the adsorp and desorp values as numpy arrays, while AIF2 collects those values into pandas DataFrames.

1. Basic metadata fields "style" that gives the record a style name and "modelroot" which is the primary element name in the tree-like data model format.
2. _init_values methods that contain _add_value calls to define all recognized AIF data fields, and provide metadata information for each.
3. Methods read_aif and build_aif to convert to/from the AIF format using gemmi.

In [2]:
# Import core yabadaba package which contains the main interaction capabilities
import yabadaba

# Import the AIF-specific subclasses
import yabadaba_aif

## 2. What do the AIF records let you do?

As mentioned above, the AIF record classes allow for transforming the data between the following formats

1. AIF file format
2. Python representation
3. Tree-like data representation, which can in turn be converted to/from JSON and XML
4. Flat dict of simple value fields

### 2.1. Loading records

Records can be loaded in from AIF, JSON or XML files.

In [3]:
aif_file_name = 'aif_database2/aif/CH4_CuBTC_300K.aif'

# Load record from AIF files - first term is "style" 
aif1 = yabadaba.load_record('aif', aif=aif_file_name)
aif2 = yabadaba.load_record('aif2', aif=aif_file_name)

# Show that the two records are of the same content, but one is AIF and the other AIF2
print(aif1)
print(aif2)

aif record named CH4_CuBTC_300K
aif2 record named CH4_CuBTC_300K


### 2.2. Python representation

All values defined in the AIF records can be accessed and set as class attributes.

In [4]:
aif1.exptl_adsorptive_name

'Methane'

AIF represents the adsorp and desorp fields as numpy arrays

In [6]:
aif1.adsorp_fugacity

array([3.723e-04, 4.444e-04, 5.305e-04, 6.332e-04, 7.559e-04, 9.022e-04,
       1.077e-03, 1.286e-03, 1.534e-03, 1.832e-03, 2.186e-03, 2.610e-03,
       3.115e-03, 3.718e-03, 4.438e-03, 5.298e-03, 6.324e-03, 7.549e-03,
       9.010e-03, 1.076e-02, 1.284e-02, 1.532e-02, 1.829e-02, 2.183e-02,
       2.606e-02, 3.111e-02, 3.713e-02, 4.433e-02, 5.291e-02, 6.316e-02,
       7.539e-02, 8.998e-02, 1.074e-01, 1.282e-01, 1.530e-01, 1.827e-01,
       2.181e-01, 2.603e-01, 3.107e-01, 3.709e-01, 4.427e-01, 5.284e-01,
       6.307e-01, 7.529e-01, 8.987e-01, 1.073e+00, 1.280e+00, 1.528e+00,
       1.824e+00, 2.178e+00, 2.599e+00, 3.103e+00, 3.704e+00, 4.421e+00,
       5.277e+00, 6.299e+00, 7.519e+00, 8.975e+00, 1.071e+01, 1.279e+01,
       1.526e+01, 1.822e+01, 2.175e+01, 2.596e+01, 3.099e+01, 3.699e+01,
       4.415e+01, 5.270e+01, 6.290e+01, 7.509e+01, 8.963e+01, 1.070e+02,
       1.277e+02, 1.524e+02, 1.820e+02, 2.172e+02, 2.592e+02, 3.095e+02,
       3.694e+02, 4.409e+02, 5.263e+02, 6.282e+02, 

AIF2 represents the adsorp and desorp fields as pandas DataFrames

In [8]:
aif2.adsorp

,fugacity,pressure,amount,amount_uncertainty,qst,qst_uncertainty
0,0.000372,0.000372,0.000378,0.000011,19.04,0.05062
1,0.000444,0.000444,0.000451,0.000013,19.04,0.05059
2,0.000531,0.000531,0.000538,0.000016,19.03,0.05057
3,0.000633,0.000633,0.000643,0.000019,19.03,0.05053
4,0.000756,0.000756,0.000767,0.000023,19.03,0.05049
...,...,...,...,...,...,...
95,7489.000000,1958.000000,18.890000,0.005472,13.69,0.16340
96,8939.000000,2067.000000,19.040000,0.005881,13.37,0.15450
97,10670.000000,2177.000000,19.180000,0.004412,13.04,0.15060
98,12740.000000,2289.000000,19.320000,0.002047,12.73,0.13520


### 2.3. Converting to AIF

Records can be converted to AIF using the build_aif method.  This can be used to modify fields in existing records, or new records can be built up in the Python representation then saved.

build_aif takes parameters
- "output_file" allows for specifying a file to save to. If not given, the AIF file will be returned as a string.
- "loop_fmt" allows for control over the float formatting of the looped values using c-style formatting code.  Default is '%.5E'.

In [20]:
# Generate the AIF file format and show the first 700 characters...
print(aif1.build_aif()[:700])

data_CH4_CuBTC_300K
_exptl_adsorptive_name Methane
_exptl_date 2023-08-09
_exptl_isotherm_type absolute
_exptl_temperature 300.0
_adsnt_material_id CuBTC
_simltn_forcefield_adsorbent 'see adsorbent file'
_simltn_forcefield_adsorptive 'TraPPE CH4; 12ang linear-force shift'
_simltn_adsorbent_filename 'data.CuBTC_CaleroIV_rep111'
_units_loading MOL-PER-KiloGM
_units_pressure BAR
_units_temperature K
_units_fugacity BAR
_units_qst KiloJ-PER-MOL

loop_
_adsorp_amount
_adsorp_amount_uncertainty
_adsorp_fugacity
_adsorp_pressure
_adsorp_qst
_adsorp_qst_uncertainty
3.77900E-04 1.13000E-05 3.72300E-04 3.72300E-04 1.90400E+01 5.06200E-02
4.51000E-04 1.34800E-05 4.44400E-04 4.44400E-04 1.90400E+01 5.05


Here's a partial example of AIF file construction

In [22]:
# Create a new empty record and give it a name
new_aif = yabadaba.load_record('aif', name='Example_AIF_Build')

# Directly assign any values
new_aif.exptl_operator = 'Stanley Ipcus'
new_aif.exptl_adsorptive_name = 'Asdf'

new_aif.adsorp_amount = [0.1, 0.3, 0.5, 0.7, 0.9]
new_aif.adsorp_fugacity = [1.235e-4, 1.683e-4, 2.142e-4, 2.8547e-4, 3.6823e-4]

# Convert to AIF format
print(new_aif.build_aif())

data_Example_AIF_Build
_exptl_adsorptive_name Asdf
_exptl_operator 'Stanley Ipcus'

loop_
_adsorp_amount
_adsorp_fugacity
1.00000E-01 1.23500E-04
3.00000E-01 1.68300E-04
5.00000E-01 2.14200E-04
7.00000E-01 2.85470E-04
9.00000E-01 3.68230E-04



### 2.4. Flat dict representation

The metadata() method generates the flat dict representation containing all simple values. **Note** that it doesn't contain adsorp or desorp fields as they are complex!

In [5]:
aif1.metadata()

{'name': 'CH4_CuBTC_300K',
 'audit_aif_version': None,
 'exptl_adsorptive': None,
 'exptl_adsorptive_name': 'Methane',
 'exptl_date': '2023-08-09',
 'exptl_digitizer': None,
 'exptl_instrument': None,
 'exptl_isotherm_type': 'absolute',
 'exptl_method': None,
 'exptl_operator': None,
 'exptl_temperature': 300.0,
 'adsnt_degas_summary': None,
 'adsnt_hashkey': None,
 'adsnt_info': None,
 'adsnt_material_id': 'CuBTC',
 'adsnt_sample_id': None,
 'simltn_code': None,
 'simltn_date': None,
 'simltn_forcefield_adsorbent': 'see adsorbent file',
 'simltn_forcefield_adsorptive': 'TraPPE CH4; 12ang linear-force shift',
 'simltn_input_files': None,
 'simltn_lot': None,
 'simltn_sampling': None,
 'simltn_size': None,
 'simltn_adsorbent_filename': 'data.CuBTC_CaleroIV_rep111',
 'simltn_forcefield_adsnt': None,
 'units_composition_type': None,
 'units_density': None,
 'units_loading': 'MOL-PER-KiloGM',
 'units_mass': None,
 'units_pressure': 'BAR',
 'units_temperature': 'K',
 'units_time': None,
 'u

### 2.5. Data model, JSON and XML representations

The tree-like data model representation can also be interacted with.

- model is the representation as a DataModelDict object (dict plus conversion and search capabilities). It automatically exists if content is loaded from JSON or XML.
- build_model() generates the model based on the current Python values.
- load_model() reads in model contents in JSON or XML format, which sets the model and Python values.
- reload_model() updates the Python values based on any changes made to model.

In [27]:
# Build the model representation
aif1.build_model()

# Print out the first 1000 charaters in JSON format
print(aif1.model.json(indent=4)[:1000])

{
    "aif": {
        "exptl": {
            "adsorptive_name": "Methane",
            "date": "2023-08-09",
            "isotherm_type": "absolute",
            "temperature": 300.0
        },
        "adsnt": {
            "material_id": "CuBTC"
        },
        "simltn": {
            "forcefield_adsorbent": "see adsorbent file",
            "forcefield_adsorptive": "TraPPE CH4; 12ang linear-force shift",
            "adsorbent_filename": "data.CuBTC_CaleroIV_rep111"
        },
        "units": {
            "loading": "MOL-PER-KiloGM",
            "pressure": "BAR",
            "temperature": "K",
            "fugacity": "BAR",
            "qst": "KiloJ-PER-MOL"
        },
        "adsorp": {
            "amount": {
                "value": [
                    0.0003779,
                    0.000451,
                    0.0005383,
                    0.0006426,
                    0.000767,
                    0.0009155,
                    0.001093,
                    0.0013

### 2.6 Automatic documentation

If you add description fields to the value definitions in the record, you can automatically generate documentation for all recognized AIF value fields.

In [25]:
print(aif1.valuedoc)

# aif Values

- __audit_aif_version__: version of AIF data names (Github commit hash)
- __exptl_adsorptive__: name of the adsorptive
- __exptl_adsorptive_name__: secondary identifier
- __exptl_date__: date of the experiment (string in ISO 8601 format)
- __exptl_digitizer__: name of the person who digitized the experiment
- __exptl_instrument__: instrument id used for the experiment
- __exptl_isotherm_type__: description of isotherm type, eg. absolute, excess, net
- __exptl_method__: description of method used to determine amount adsorbed, eg. volumetric
- __exptl_operator__: name of the person who ran the experiment
- __exptl_p0__: saturation pressure of the experiment at the temperature of the experiment
- __exptl_p0_uncertainty__: saturation pressure of the experiment at the temperature of the experiment uncertainty
- __exptl_temperature__: temperature of the experiment
- __exptl_temperature_uncertainty__: temperature of the experiment uncertainty
- __adsnt_degas_summary__: summary o